# 图文检索系统实战教程

本教程深入讲解如何构建高效的图文检索系统：
- 图像到文本检索 (Image-to-Text)
- 文本到图像检索 (Text-to-Image)
- 向量索引与加速
- 重排序策略

In [ ]:
import sys
sys.path.insert(0, '../src')

import torch
import torch.nn.functional as F
from typing import List, Dict, Tuple, Optional
import numpy as np
from dataclasses import dataclass

from clip import CLIP, create_clip_model

## 1. 图文检索基础

### 检索原理

```
图像检索文本:  Query(Image) → CLIP → 图像特征 → 与文本库匹配 → Top-K 文本
文本检索图像:  Query(Text)  → CLIP → 文本特征 → 与图像库匹配 → Top-K 图像
```

In [ ]:
@dataclass
class RetrievalResult:
    """检索结果"""
    indices: List[int]
    scores: List[float]
    
class ImageTextRetriever:
    """
    图文检索系统
    
    支持:
    - 图像到文本检索
    - 文本到图像检索
    - 批量检索
    """
    
    def __init__(self, model: CLIP):
        self.model = model
        self.image_features = None
        self.text_features = None
        self.image_ids = []
        self.text_ids = []
    
    def build_image_index(self, images: torch.Tensor, image_ids: List[str] = None):
        """构建图像索引"""
        with torch.no_grad():
            features = self.model.encode_image(images)
            self.image_features = F.normalize(features, dim=-1)
        
        self.image_ids = image_ids or [f"img_{i}" for i in range(len(images))]
        print(f"Built image index with {len(self.image_ids)} images")
    
    def build_text_index(self, input_ids: torch.Tensor, text_ids: List[str] = None):
        """构建文本索引"""
        with torch.no_grad():
            features = self.model.encode_text(input_ids)
            self.text_features = F.normalize(features, dim=-1)
        
        self.text_ids = text_ids or [f"text_{i}" for i in range(len(input_ids))]
        print(f"Built text index with {len(self.text_ids)} texts")
    
    def image_to_text(self, query_image: torch.Tensor, top_k: int = 5) -> RetrievalResult:
        """图像检索文本"""
        with torch.no_grad():
            query_feat = self.model.encode_image(query_image)
            query_feat = F.normalize(query_feat, dim=-1)
            
            scores = query_feat @ self.text_features.T
            scores = scores.squeeze(0)
            
            top_scores, top_indices = scores.topk(top_k)
        
        return RetrievalResult(
            indices=top_indices.tolist(),
            scores=top_scores.tolist()
        )
    
    def text_to_image(self, query_text: torch.Tensor, top_k: int = 5) -> RetrievalResult:
        """文本检索图像"""
        with torch.no_grad():
            query_feat = self.model.encode_text(query_text)
            query_feat = F.normalize(query_feat, dim=-1)
            
            scores = query_feat @ self.image_features.T
            scores = scores.squeeze(0)
            
            top_scores, top_indices = scores.topk(top_k)
        
        return RetrievalResult(
            indices=top_indices.tolist(),
            scores=top_scores.tolist()
        )

# 测试
model = create_clip_model("small")
retriever = ImageTextRetriever(model)

# 构建索引
images = torch.randn(100, 3, 224, 224)
texts = torch.randint(0, 49408, (100, 77))
retriever.build_image_index(images)
retriever.build_text_index(texts)

# 检索
query_img = torch.randn(1, 3, 224, 224)
result = retriever.image_to_text(query_img, top_k=5)
print(f"Top-5 text indices: {result.indices}")
print(f"Top-5 scores: {[f'{s:.4f}' for s in result.scores]}")

## 2. 向量索引加速

对于大规模检索，需要使用近似最近邻 (ANN) 算法加速。

In [ ]:
class ApproximateNNIndex:
    """
    近似最近邻索引 (简化实现)
    
    实际应用中可使用:
    - FAISS (Facebook AI Similarity Search)
    - Annoy (Spotify)
    - ScaNN (Google)
    """
    
    def __init__(self, dim: int, n_clusters: int = 100):
        self.dim = dim
        self.n_clusters = n_clusters
        self.centroids = None
        self.cluster_assignments = None
        self.vectors = None
    
    def build(self, vectors: torch.Tensor, n_iter: int = 10):
        """构建索引 (简化的 K-Means 聚类)"""
        self.vectors = vectors
        n_vectors = vectors.shape[0]
        
        # 随机初始化聚类中心
        indices = torch.randperm(n_vectors)[:self.n_clusters]
        self.centroids = vectors[indices].clone()
        
        # K-Means 迭代
        for _ in range(n_iter):
            # 分配到最近的聚类中心
            distances = torch.cdist(vectors, self.centroids)
            self.cluster_assignments = distances.argmin(dim=1)
            
            # 更新聚类中心
            for c in range(self.n_clusters):
                mask = self.cluster_assignments == c
                if mask.sum() > 0:
                    self.centroids[c] = vectors[mask].mean(dim=0)
        
        print(f"Built ANN index with {self.n_clusters} clusters")
    
    def search(self, query: torch.Tensor, top_k: int = 10, n_probe: int = 10) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        近似搜索
        
        Args:
            query: 查询向量
            top_k: 返回数量
            n_probe: 探测的聚类数量
        """
        # 找到最近的 n_probe 个聚类
        centroid_distances = torch.cdist(query, self.centroids)
        _, nearest_clusters = centroid_distances.topk(n_probe, largest=False)
        nearest_clusters = nearest_clusters.squeeze(0)
        
        # 在这些聚类中搜索
        candidate_mask = torch.zeros(len(self.vectors), dtype=torch.bool)
        for c in nearest_clusters:
            candidate_mask |= (self.cluster_assignments == c)
        
        candidate_indices = torch.where(candidate_mask)[0]
        candidate_vectors = self.vectors[candidate_indices]
        
        # 精确搜索候选集
        scores = query @ candidate_vectors.T
        scores = scores.squeeze(0)
        
        top_scores, top_local_indices = scores.topk(min(top_k, len(scores)))
        top_indices = candidate_indices[top_local_indices]
        
        return top_indices, top_scores

# 测试 ANN 索引
ann_index = ApproximateNNIndex(dim=256, n_clusters=10)
vectors = torch.randn(1000, 256)
vectors = F.normalize(vectors, dim=-1)
ann_index.build(vectors)

query = torch.randn(1, 256)
query = F.normalize(query, dim=-1)
indices, scores = ann_index.search(query, top_k=5)
print(f"ANN search results: {indices.tolist()}")

## 3. 重排序策略

两阶段检索：粗排 + 精排

In [ ]:
class TwoStageRetriever:
    """
    两阶段检索系统
    
    Stage 1: 粗排 - 使用 CLIP 快速召回候选
    Stage 2: 精排 - 使用更复杂的模型重排序
    """
    
    def __init__(self, clip_model: CLIP, reranker_model: Optional[CLIP] = None):
        self.clip_model = clip_model
        self.reranker = reranker_model or clip_model
        self.image_features = None
    
    def build_index(self, images: torch.Tensor):
        """构建图像索引"""
        with torch.no_grad():
            self.image_features = self.clip_model.encode_image(images)
            self.image_features = F.normalize(self.image_features, dim=-1)
        self.images = images
    
    def retrieve(self, query_text: torch.Tensor, top_k: int = 10, 
                 recall_k: int = 100) -> RetrievalResult:
        """
        两阶段检索
        
        Args:
            query_text: 查询文本
            top_k: 最终返回数量
            recall_k: 粗排召回数量
        """
        # Stage 1: 粗排
        with torch.no_grad():
            query_feat = self.clip_model.encode_text(query_text)
            query_feat = F.normalize(query_feat, dim=-1)
            
            coarse_scores = query_feat @ self.image_features.T
            coarse_scores = coarse_scores.squeeze(0)
            _, candidate_indices = coarse_scores.topk(recall_k)
        
        # Stage 2: 精排
        candidate_images = self.images[candidate_indices]
        
        with torch.no_grad():
            # 使用更精细的特征计算
            refined_img_feat = self.reranker.encode_image(candidate_images)
            refined_img_feat = F.normalize(refined_img_feat, dim=-1)
            
            refined_txt_feat = self.reranker.encode_text(query_text)
            refined_txt_feat = F.normalize(refined_txt_feat, dim=-1)
            
            fine_scores = refined_txt_feat @ refined_img_feat.T
            fine_scores = fine_scores.squeeze(0)
            
            top_scores, top_local_indices = fine_scores.topk(top_k)
            top_indices = candidate_indices[top_local_indices]
        
        return RetrievalResult(
            indices=top_indices.tolist(),
            scores=top_scores.tolist()
        )

# 测试两阶段检索
two_stage = TwoStageRetriever(model)
two_stage.build_index(images)

query = torch.randint(0, 49408, (1, 77))
result = two_stage.retrieve(query, top_k=5, recall_k=50)
print(f"Two-stage retrieval: {result.indices}")

## 4. 评估指标

In [ ]:
def compute_recall_at_k(predictions: List[List[int]], ground_truth: List[int], k: int) -> float:
    """计算 Recall@K"""
    hits = 0
    for pred, gt in zip(predictions, ground_truth):
        if gt in pred[:k]:
            hits += 1
    return hits / len(ground_truth)

def compute_mrr(predictions: List[List[int]], ground_truth: List[int]) -> float:
    """计算 Mean Reciprocal Rank"""
    rr_sum = 0
    for pred, gt in zip(predictions, ground_truth):
        if gt in pred:
            rank = pred.index(gt) + 1
            rr_sum += 1 / rank
    return rr_sum / len(ground_truth)

# 模拟评估
predictions = [[0, 1, 2, 3, 4], [5, 6, 7, 8, 9], [10, 11, 12, 13, 14]]
ground_truth = [1, 5, 15]

r1 = compute_recall_at_k(predictions, ground_truth, k=1)
r5 = compute_recall_at_k(predictions, ground_truth, k=5)
mrr = compute_mrr(predictions, ground_truth)

print(f"Recall@1: {r1:.4f}")
print(f"Recall@5: {r5:.4f}")
print(f"MRR: {mrr:.4f}")

## 总结

本教程介绍了图文检索系统的核心技术：

1. **基础检索**: 图像到文本、文本到图像的双向检索
2. **向量索引**: 使用 ANN 算法加速大规模检索
3. **两阶段检索**: 粗排召回 + 精排重排序
4. **评估指标**: Recall@K, MRR 等标准指标